In [3]:

import requests
import pandas as pd

from io import StringIO

english_url = "https://raw.githubusercontent.com/nlpc-uom/English-Tamil-Parallel-Corpus/master/En-Ta%20Corpus/En-Ta%20English.txt"
tamil_url = "https://raw.githubusercontent.com/nlpc-uom/English-Tamil-Parallel-Corpus/master/En-Ta%20Corpus/En-Ta%20Tamil.txt"

print("Downloading English dataset...")
english_response = requests.get(english_url)
english_response.raise_for_status()

print("Downloading Tamil dataset...")
tamil_response = requests.get(tamil_url)
tamil_response.raise_for_status()

english_lines = english_response.text.splitlines()
tamil_lines = tamil_response.text.splitlines()

print(f"Downloaded English lines: {len(english_lines)}")
print(f"Downloaded Tamil lines:   {len(tamil_lines)}")

english_lines = english_lines[3:]
tamil_lines = tamil_lines[3:]

num_pairs = min(len(english_lines), len(tamil_lines))

english_lines = english_lines[:num_pairs]
tamil_lines = tamil_lines[:num_pairs]

df = pd.DataFrame({
    "english": english_lines,
    "tamil": tamil_lines
})

df["english"] = df["english"].astype(str).str.strip()
df["tamil"] = df["tamil"].astype(str).str.strip()

df = df[
    (df["english"] != "") &
    (df["tamil"] != "")
]

df = df.drop_duplicates().reset_index(drop=True)

output_file = "English-Tamil-Parallel-Corpus.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\n==========================================")
print("DATASET CREATED SUCCESSFULLY")
print("==========================================")
print(f"File name : {output_file}")
print(f"Rows      : {len(df)}")
print(f"Columns   : {list(df.columns)}")

display(df.head(10))


Downloaded English lines: 8949
Downloaded Tamil lines:   8949

DATASET CREATED SUCCESSFULLY
File name : English-Tamil-Parallel-Corpus.csv
Rows      : 6869
Columns   : ['english', 'tamil']


,english,tamil
0,Ranaviru Sewa Authority,ரணவிரு சேவை அதிகார சபை
1,Annual Report 2011,வருடாந்த அறிக்கை 2011
2,"No.301, 4th Floor,","இல. 301, 4ஆம் மாடி,"
3,T.B.Jayah Mawatha,டி.பி. ஜாயா மாவத்தை
4,Colombo 10,கொழும்பு 10
5,Contents,உள்ளடக்கம்
6,01. Nation’s gratitude to the name 'Ranaviru',01. இராணுவ வீரன் எனும் பெயருக்கு தேசத்தின் நன்றி
7,02. Preamble,02. முன்னுரை
8,03. Our Functions,03. எமது பணிகள்
9,04. Targeted Beneficiaries,04. இலக்காக கொள்ளப்பட்ட பயனாளிகள்


In [ ]:
# ============================================================
# ENGLISH → TAMIL ATTENTION-BASED NEURAL MACHINE TRANSLATOR
# ============================================================

# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import random
import math

print("PyTorch version:", torch.__version__)


# ------------------------------------------------------------
# 2. DEVICE
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ------------------------------------------------------------
# 3. RANDOM SEED
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# ------------------------------------------------------------
# 4. LOAD DATASET
# ------------------------------------------------------------

CSV_FILE = "english_tamil_parallel_corpus.csv"

df = pd.read_csv(
    CSV_FILE,
    encoding="utf-8-sig"
)

print("\nDataset loaded successfully!")
print("Dataset size:", len(df))

print("\nFirst 5 rows:")
print(df.head())


# ------------------------------------------------------------
# 5. CLEAN DATA
# ------------------------------------------------------------

df = df.dropna(
    subset=["english", "tamil"]
)

df["english"] = (
    df["english"]
    .astype(str)
    .str.strip()
)

df["tamil"] = (
    df["tamil"]
    .astype(str)
    .str.strip()
)

print(
    "\nDataset size after cleaning:",
    len(df)
)


# ------------------------------------------------------------
# 6. TOKENIZATION
# ------------------------------------------------------------

def tokenize(sentence):

    return (
        sentence
        .lower()
        .strip()
        .split()
    )


# ------------------------------------------------------------
# 7. SPECIAL TOKENS
# ------------------------------------------------------------

PAD_TOKEN = "<pad>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"


# ------------------------------------------------------------
# 8. BUILD VOCABULARY
# ------------------------------------------------------------

def build_vocab(sentences):

    vocab = {
        PAD_TOKEN: 0,
        SOS_TOKEN: 1,
        EOS_TOKEN: 2,
        UNK_TOKEN: 3
    }

    for sentence in sentences:

        tokens = tokenize(sentence)

        for token in tokens:

            if token not in vocab:

                vocab[token] = len(vocab)

    return vocab


english_vocab = build_vocab(
    df["english"]
)

tamil_vocab = build_vocab(
    df["tamil"]
)


# Reverse vocabulary

english_itos = {
    index: word
    for word, index in english_vocab.items()
}

tamil_itos = {
    index: word
    for word, index in tamil_vocab.items()
}


print("\nEnglish vocabulary:",
      len(english_vocab))

print("Tamil vocabulary:",
      len(tamil_vocab))


# ------------------------------------------------------------
# 9. NUMERICALIZATION
# ------------------------------------------------------------

def numericalize(
    sentence,
    vocab
):

    tokens = tokenize(sentence)

    ids = []

    # Start token
    ids.append(
        vocab[SOS_TOKEN]
    )

    for token in tokens:

        if token in vocab:

            ids.append(
                vocab[token]
            )

        else:

            ids.append(
                vocab[UNK_TOKEN]
            )

    # End token
    ids.append(
        vocab[EOS_TOKEN]
    )

    return ids


# ------------------------------------------------------------
# 10. CREATE DATA
# ------------------------------------------------------------

data = list(
    zip(
        df["english"],
        df["tamil"]
    )
)

random.shuffle(data)


# ------------------------------------------------------------
# 11. TRAIN / TEST SPLIT
# ------------------------------------------------------------

split_index = int(
    len(data) * 0.8
)

train_data = data[
    :split_index
]

test_data = data[
    split_index:
]

print("\nTraining samples:",
      len(train_data))

print("Testing samples:",
      len(test_data))


# ------------------------------------------------------------
# 12. ENCODER
# ------------------------------------------------------------

class Encoder(nn.Module):

    def __init__(
        self,
        input_dim,
        embedding_dim,
        hidden_dim
    ):

        super().__init__()

        # Word embedding
        self.embedding = nn.Embedding(
            input_dim,
            embedding_dim
        )

        # GRU
        self.rnn = nn.GRU(
            embedding_dim,
            hidden_dim
        )


    def forward(self, src):

        # src:
        # [source_length, batch_size]

        embedded = self.embedding(src)

        # embedded:
        # [source_length,
        #  batch_size,
        #  embedding_dim]

        outputs, hidden = self.rnn(
            embedded
        )

        # outputs:
        # [source_length,
        #  batch_size,
        #  hidden_dim]

        # hidden:
        # [1,
        #  batch_size,
        #  hidden_dim]

        return outputs, hidden


# ------------------------------------------------------------
# 13. ATTENTION MECHANISM
# ------------------------------------------------------------

class Attention(nn.Module):

    def __init__(
        self,
        hidden_dim
    ):

        super().__init__()

        self.attention = nn.Linear(
            hidden_dim * 2,
            hidden_dim
        )

        self.v = nn.Linear(
            hidden_dim,
            1,
            bias=False
        )


    def forward(
        self,
        hidden,
        encoder_outputs
    ):

        # hidden:
        # [1, batch_size, hidden_dim]

        # encoder_outputs:
        # [source_length,
        #  batch_size,
        #  hidden_dim]

        src_len = (
            encoder_outputs.shape[0]
        )


        # Get last hidden state
        hidden = hidden[-1]

        # hidden:
        # [batch_size, hidden_dim]


        # Add source sequence dimension
        hidden = hidden.unsqueeze(0)

        # hidden:
        # [1, batch_size, hidden_dim]


        # Repeat for every source word
        hidden = hidden.repeat(
            src_len,
            1,
            1
        )

        # hidden:
        # [source_length,
        #  batch_size,
        #  hidden_dim]


        # Combine:
        #
        # decoder hidden state
        #
        # +
        #
        # encoder output

        energy = torch.tanh(
            self.attention(
                torch.cat(
                    (
                        hidden,
                        encoder_outputs
                    ),
                    dim=2
                )
            )
        )


        # energy:
        # [source_length,
        #  batch_size,
        #  hidden_dim]


        # Calculate attention score
        attention = self.v(
            energy
        ).squeeze(2)


        # attention:
        # [source_length,
        #  batch_size]


        # Change dimensions
        attention = attention.permute(
            1,
            0
        )


        # attention:
        # [batch_size,
        #  source_length]


        # Normalize scores
        attention = torch.softmax(
            attention,
            dim=1
        )


        return attention


# ------------------------------------------------------------
# 14. DECODER
# ------------------------------------------------------------

class Decoder(nn.Module):

    def __init__(
        self,
        output_dim,
        embedding_dim,
        hidden_dim,
        attention
    ):

        super().__init__()

        self.output_dim = output_dim

        self.attention = attention


        # Tamil embedding
        self.embedding = nn.Embedding(
            output_dim,
            embedding_dim
        )


        # Decoder GRU
        self.rnn = nn.GRU(
            embedding_dim + hidden_dim,
            hidden_dim
        )


        # Output layer
        self.fc_out = nn.Linear(
            embedding_dim
            + hidden_dim
            + hidden_dim,
            output_dim
        )


    def forward(
        self,
        input,
        hidden,
        encoder_outputs
    ):

        # input:
        # [batch_size]


        # Add sequence dimension
        input = input.unsqueeze(0)

        # [1, batch_size]


        # Embedding
        embedded = self.embedding(
            input
        )

        # [1,
        #  batch_size,
        #  embedding_dim]


        # Calculate attention
        attention_weights = self.attention(
            hidden,
            encoder_outputs
        )

        # [batch_size,
        #  source_length]


        # Add dimension for matrix multiplication
        attention_weights = (
            attention_weights.unsqueeze(1)
        )

        # [batch_size,
        #  1,
        #  source_length]


        # Rearrange encoder outputs
        encoder_outputs = (
            encoder_outputs.permute(
                1,
                0,
                2
            )
        )

        # [batch_size,
        #  source_length,
        #  hidden_dim]


        # Calculate context vector
        weighted = torch.bmm(
            attention_weights,
            encoder_outputs
        )

        # [batch_size,
        #  1,
        #  hidden_dim]


        # Rearrange
        weighted = weighted.permute(
            1,
            0,
            2
        )

        # [1,
        #  batch_size,
        #  hidden_dim]


        # Combine embedding + context
        rnn_input = torch.cat(
            (
                embedded,
                weighted
            ),
            dim=2
        )


        # Decoder GRU
        output, hidden = self.rnn(
            rnn_input,
            hidden
        )


        # Remove sequence dimension

        embedded = embedded.squeeze(0)

        output = output.squeeze(0)

        weighted = weighted.squeeze(0)


        # Final prediction
        prediction = self.fc_out(
            torch.cat(
                (
                    output,
                    weighted,
                    embedded
                ),
                dim=1
            )
        )


        return (
            prediction,
            hidden
        )


# ------------------------------------------------------------
# 15. SEQ2SEQ MODEL
# ------------------------------------------------------------

class Seq2Seq(nn.Module):

    def __init__(
        self,
        encoder,
        decoder,
        device
    ):

        super().__init__()

        self.encoder = encoder

        self.decoder = decoder

        self.device = device


    def forward(
        self,
        src,
        trg,
        teacher_forcing_ratio=0.5
    ):

        batch_size = src.shape[1]

        trg_len = trg.shape[0]

        trg_vocab_size = (
            self.decoder.output_dim
        )


        # Store predictions
        outputs = torch.zeros(
            trg_len,
            batch_size,
            trg_vocab_size
        ).to(self.device)


        # Encoder
        encoder_outputs, hidden = (
            self.encoder(src)
        )


        # First input = <sos>
        input = trg[0, :]


        # Decode
        for t in range(
            1,
            trg_len
        ):

            output, hidden = (
                self.decoder(
                    input,
                    hidden,
                    encoder_outputs
                )
            )


            # Store output
            outputs[t] = output


            # Teacher forcing
            teacher_force = (
                random.random()
                < teacher_forcing_ratio
            )


            # Most probable token
            top1 = output.argmax(
                1
            )


            if teacher_force:

                input = trg[t]

            else:

                input = top1


        return outputs


# ------------------------------------------------------------
# 16. MODEL PARAMETERS
# ------------------------------------------------------------

INPUT_DIM = len(
    english_vocab
)

OUTPUT_DIM = len(
    tamil_vocab
)

ENC_EMB_DIM = 128

DEC_EMB_DIM = 128

HID_DIM = 256


# ------------------------------------------------------------
# 17. CREATE ATTENTION
# ------------------------------------------------------------

attention = Attention(
    HID_DIM
)


# ------------------------------------------------------------
# 18. CREATE ENCODER
# ------------------------------------------------------------

encoder = Encoder(
    INPUT_DIM,
    ENC_EMB_DIM,
    HID_DIM
)


# ------------------------------------------------------------
# 19. CREATE DECODER
# ------------------------------------------------------------

decoder = Decoder(
    OUTPUT_DIM,
    DEC_EMB_DIM,
    HID_DIM,
    attention
)


# ------------------------------------------------------------
# 20. CREATE SEQ2SEQ MODEL
# ------------------------------------------------------------

model = Seq2Seq(
    encoder,
    decoder,
    device
).to(device)


print("\nModel created successfully!")


# ------------------------------------------------------------
# 21. OPTIMIZER
# ------------------------------------------------------------

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


# ------------------------------------------------------------
# 22. LOSS FUNCTION
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss(
    ignore_index=tamil_vocab[
        PAD_TOKEN
    ]
)


# ------------------------------------------------------------
# 23. CONVERT SENTENCE TO TENSOR
# ------------------------------------------------------------

def sentence_to_tensor(
    sentence,
    vocab
):

    ids = numericalize(
        sentence,
        vocab
    )


    tensor = torch.tensor(
        ids,
        dtype=torch.long
    )


    # Add batch dimension
    tensor = tensor.unsqueeze(1)


    return tensor.to(device)


# ------------------------------------------------------------
# 24. TRAINING FUNCTION
# ------------------------------------------------------------

def train_one_epoch():

    model.train()

    total_loss = 0


    # Shuffle training data
    random.shuffle(train_data)


    for english, tamil in train_data:


        # Convert English
        src = sentence_to_tensor(
            english,
            english_vocab
        )


        # Convert Tamil
        trg = sentence_to_tensor(
            tamil,
            tamil_vocab
        )


        # Clear gradients
        optimizer.zero_grad()


        # Forward pass
        output = model(
            src,
            trg,
            teacher_forcing_ratio=0.5
        )


        # Remove <sos>
        output = output[1:]


        # Remove batch dimension
        output = output.squeeze(1)


        # Remove <sos> from target
        trg = trg[1:]

        trg = trg.squeeze(1)


        # Calculate loss
        loss = criterion(
            output,
            trg
        )


        # Backpropagation
        loss.backward()


        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1
        )


        # Update weights
        optimizer.step()


        total_loss += loss.item()


    return (
        total_loss /
        len(train_data)
    )


# ------------------------------------------------------------
# 25. TRAIN MODEL
# ------------------------------------------------------------

EPOCHS = 200


print("\n")
print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)


for epoch in range(
    EPOCHS
):

    loss = train_one_epoch()


    if (
        (epoch + 1) % 10
        == 0
    ):

        print(
            f"Epoch "
            f"{epoch + 1:03d}/{EPOCHS} "
            f"| Loss: {loss:.4f}"
        )


print("\nTraining completed!")


# ------------------------------------------------------------
# 26. TRANSLATION FUNCTION
# ------------------------------------------------------------

def translate_sentence(
    sentence
):

    model.eval()


    # Convert English sentence
    src = sentence_to_tensor(
        sentence,
        english_vocab
    )


    # Encoder
    with torch.no_grad():

        encoder_outputs, hidden = (
            model.encoder(src)
        )


    # Start with <sos>
    input = torch.tensor(
        [
            tamil_vocab[
                SOS_TOKEN
            ]
        ],
        dtype=torch.long
    ).to(device)


    translated_words = []


    # Maximum output length
    for _ in range(30):


        # Decoder
        with torch.no_grad():

            output, hidden = (
                model.decoder(
                    input,
                    hidden,
                    encoder_outputs
                )
            )


        # Most probable token
        predicted_token = (
            output.argmax(1).item()
        )


        # Stop at <eos>
        if (
            predicted_token
            == tamil_vocab[EOS_TOKEN]
        ):

            break


        # Convert number → Tamil word
        predicted_word = (
            tamil_itos[
                predicted_token
            ]
        )


        # Ignore special tokens
        if predicted_word not in [
            PAD_TOKEN,
            SOS_TOKEN,
            EOS_TOKEN
        ]:

            translated_words.append(
                predicted_word
            )


        # Next decoder input
        input = torch.tensor(
            [predicted_token],
            dtype=torch.long
        ).to(device)


    # Join Tamil words
    return " ".join(
        translated_words
    )


# ------------------------------------------------------------
# 27. TEST MODEL ON EXAMPLES
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("ENGLISH → TAMIL TRANSLATION")
print("=" * 60)


test_sentences = [

    "hello",

    "good morning",

    "good evening",

    "thank you",

    "how are you?",

    "I am fine.",

    "I am studying.",

    "I like coffee.",

    "Where are you going?",

    "I am going to college.",

    "Do you speak Tamil?"

]


for sentence in test_sentences:

    translation = (
        translate_sentence(
            sentence
        )
    )


    print(
        "\nEnglish:",
        sentence
    )

    print(
        "Tamil  :",
        translation
    )


# ------------------------------------------------------------
# 28. TEST WITH USER INPUT
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("INTERACTIVE TRANSLATOR")
print("=" * 60)

print(
    "Type 'exit' to stop."
)


while True:

    sentence = input(
        "\nEnter English sentence: "
    )


    if sentence.lower() == "exit":

        print(
            "Translator stopped."
        )

        break


    if not sentence.strip():

        print(
            "Please enter a sentence."
        )

        continue


    translation = (
        translate_sentence(
            sentence
        )
    )


    print(
        "Tamil:",
        translation
    )


# ------------------------------------------------------------
# 29. SAVE MODEL
# ------------------------------------------------------------

torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "english_vocab":
            english_vocab,

        "tamil_vocab":
            tamil_vocab
    },

    "english_tamil_attention_model.pth"
)


print(
    "\nModel saved as:"
    " english_tamil_attention_model.pth"
)

PyTorch version: 2.9.1+cpu
Using device: cpu

Dataset loaded successfully!
Dataset size: 6869

First 5 rows:
                   english                   tamil
0  Ranaviru Sewa Authority  ரணவிரு சேவை அதிகார சபை
1       Annual Report 2011   வருடாந்த அறிக்கை 2011
2       No.301, 4th Floor,     இல. 301, 4ஆம் மாடி,
3        T.B.Jayah Mawatha     டி.பி. ஜாயா மாவத்தை
4               Colombo 10             கொழும்பு 10

Dataset size after cleaning: 6869

English vocabulary: 9167
Tamil vocabulary: 15279

Training samples: 5495
Testing samples: 1374

Model created successfully!


STARTING TRAINING
